In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# =========================
# Import REIT Prices (same timeframe: 2025-01-01 to 2025-12-31)
# =========================
start = pd.Timestamp("2025-01-01").date()
end = pd.Timestamp("2025-12-31").date()


def fetch_one(symbols, start, end):
    """Try a list of symbols; return the first non-empty Close series (tz-naive)."""
    for s in symbols:
        try:
            df = yf.download(
                s,
                start=start,
                end=end,
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            if isinstance(df, pd.DataFrame) and "Close" in df and not df["Close"].dropna().empty:
                sr = df["Close"].dropna().copy()
                sr.index = pd.to_datetime(sr.index).tz_localize(None)
                sr.name = s
                return sr
        except Exception:
            pass
    return pd.Series(dtype=float)


# Major US REITs (office, residential, industrial, retail mix)
pld = fetch_one(["PLD"], start, end)  # Prologis (industrial)
pld.name = "PLD"

vno = fetch_one(["VNO"], start, end)  # Vornado (office/retail, NYC-focused)
vno.name = "VNO"

eqr = fetch_one(["EQR"], start, end)  # Equity Residential
eqr.name = "EQR"

realty_income = fetch_one(["O"], start, end)  # Realty Income
realty_income.name = "O"

# Assemble REIT prices
df_reits = pd.concat([pld, vno, eqr, realty_income], axis=1).dropna(how="all")

print("REIT Prices (2025):")
display(df_reits.tail())

# =========================
# Prepare for Market Model Analysis
# =========================
# coerce to clean daily Series (tz-naive)
def to_clean_daily_series(x) -> pd.Series:
    if isinstance(x, pd.DataFrame):
        if x.shape[1] != 1:
            raise ValueError("Expected a single-column DataFrame for the composite index.")
        x = x.iloc[:, 0]
    s = x.copy()
    s.index = pd.to_datetime(s.index)
    if s.index.tz is not None:
        s.index = s.index.tz_convert("UTC").tz_localize(None)
    s = s.sort_index().astype(float)
    s = s.resample("D").last().ffill()
    s.name = s.name or "INDEX"
    return s


# Build a simple equal-weight REIT portfolio for analysis
reit_prices = df_reits[["PLD", "VNO", "EQR", "O"]].dropna()
reit_composite = reit_prices.mean(axis=1)
reit_composite.name = "REIT_EW"

# Log returns
reit_ret = np.log(reit_composite / reit_composite.shift(1)).dropna()

print(f"\nREIT composite return coverage: {reit_ret.index.min().date()} → {reit_ret.index.max().date()} ({len(reit_ret)} days)")

REIT Prices (2025):


Ticker,PLD,VNO,EQR,O
Date,,,,
2025-12-23,127.769997,33.009998,61.644810,55.417931
2025-12-24,129.149994,33.650002,62.040409,56.151154
2025-12-26,128.710007,33.650002,62.396450,56.170971
2025-12-29,128.479996,33.540001,62.584362,56.289875
2025-12-30,129.009995,33.689999,62.801945,56.507858



REIT composite return coverage: 2025-01-03 → 2025-12-30 (248 days)


In [2]:
from pathlib import Path
import pandas as pd

csv_path = Path("NYC.csv")
argentina_csv = pd.read_csv(csv_path)

display(argentina_csv.head())

,category,Jim Walden,Curtis Sliwa,Brad Lander,Eric Adams,Andrew Yang,Zohran Mamdani,Scott Stringer,Rudy Giuliani,Zellnor Myrie,Andrew Cuomo,Adrienne Adams,Michael Bloomberg
0,Tue Apr 22 2025,0.010,0.010,0.004,0.015,0.004,0.09,0.004,0.011,0.004,0.77,0.003,NaN
1,Wed Apr 23 2025,NaN,NaN,0.010,0.021,NaN,0.08,NaN,NaN,NaN,0.77,0.005,NaN
2,Thu Apr 24 2025,0.022,0.022,0.010,0.025,0.016,0.08,0.017,0.016,0.011,0.78,0.011,NaN
3,Fri Apr 25 2025,NaN,NaN,NaN,NaN,NaN,0.10,NaN,NaN,NaN,NaN,NaN,NaN
4,Sat Apr 26 2025,NaN,NaN,NaN,0.025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
